# Deep CFR GPU Reference Run

This is the production reference run for `r4_s4_h2_hp2pt_ss` using the finalized split architecture:

- advantage networks: `2048x2048`
- average-strategy networks: `512x512`
- GPU-native traversal and device-resident replay
- exact learned-average evaluation every five measured training minutes
- exact-resume checkpoint every fifteen measured training minutes

Exact evaluation runs in one CPU worker and does not count against the 45-minute GPU-training budget.

In [ ]:
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.core import GameSpec
from liars_poker.eval.deep_cfr_async import AsyncDeepCFREvaluator
from liars_poker.serialization import save_policy

assert torch.cuda.is_available(), 'This reference run requires CUDA.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CPU count:', os.cpu_count())

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

TRAINING_MINUTES = 45
EVALUATE_EVERY_MINUTES = 5
CHECKPOINT_EVERY_MINUTES = 15
SEED = 7

TRAINER_KWARGS = {
    'advantage_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'advantage_buffer_capacity': 2_000_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'advantage_train_steps': 100,
    'strategy_train_steps': 50,
    'advantage_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'highest_regret_fallback': True,
    'alternating_updates': True,
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}
TRAVERSALS_PER_PLAYER = 1024

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs' / f'{spec.to_short_str()}___{run_id}'
CHECKPOINT_PATH = RUN_DIR / 'latest_checkpoint.pt'
FINAL_POLICY_DIR = RUN_DIR / 'policy'
TRAINING_JSONL = RUN_DIR / 'training.jsonl'
EXACT_RESULTS_JSONL = RUN_DIR / 'exact_results.jsonl'
MANIFEST_PATH = RUN_DIR / 'manifest.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    'run_type': 'deep_cfr_gpu_reference',
    'spec': spec.to_json(),
    'training_minutes': TRAINING_MINUTES,
    'evaluate_every_minutes': EVALUATE_EVERY_MINUTES,
    'checkpoint_every_minutes': CHECKPOINT_EVERY_MINUTES,
    'traversals_per_player': TRAVERSALS_PER_PLAYER,
    'trainer_kwargs': {
        **TRAINER_KWARGS,
        'device': str(device),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Run directory:', RUN_DIR)

In [ ]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'Not JSON serializable: {type(value)!r}')


def append_jsonl(path, records):
    if not records:
        return
    with path.open('a', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, default=json_default) + '\n')


def load_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def atomic_checkpoint(trainer, destination):
    temporary = destination.with_suffix(destination.suffix + '.new')
    trainer.save_checkpoint(temporary)
    os.replace(temporary, destination)


def checkpoint_summary(trainer, measured_training_s):
    return {
        'iteration': trainer.iteration,
        'measured_training_s': measured_training_s,
        'checkpoint_path': str(CHECKPOINT_PATH),
        'checkpoint_GiB': CHECKPOINT_PATH.stat().st_size / 2**30,
    }

## Run 45 measured GPU-training minutes

The latest checkpoint is replaced atomically only after the new checkpoint has been fully written. Learned-average snapshots and their exact results remain append-only.

In [ ]:
RUN_REFERENCE = True
training_records = []
checkpoint_records = []

if RUN_REFERENCE:
    trainer = DeepCFRTrainer(spec, **TRAINER_KWARGS)
    evaluator = AsyncDeepCFREvaluator(
        RUN_DIR,
        max_workers=1,
        compile_batch_size=65_536,
    )
    measured_training_s = 0.0
    next_evaluation_s = EVALUATE_EVERY_MINUTES * 60.0
    next_checkpoint_s = CHECKPOINT_EVERY_MINUTES * 60.0
    actual_start = time.perf_counter()

    try:
        while measured_training_s < TRAINING_MINUTES * 60.0:
            start = time.perf_counter()
            record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
            measured_training_s += time.perf_counter() - start
            record['measured_training_s'] = measured_training_s
            training_records.append(record)
            append_jsonl(TRAINING_JSONL, [record])

            append_jsonl(EXACT_RESULTS_JSONL, evaluator.collect_ready())

            if measured_training_s >= next_evaluation_s:
                validation = trainer.validation_metrics()
                evaluator.submit(
                    trainer.iteration,
                    trainer.average_policy(),
                    label='learned_average',
                    metadata={
                        'measured_training_s': measured_training_s,
                        'seed': SEED,
                        'advantage_validation_mse': float(np.mean([
                            player['mse'] for player in validation['advantage']
                        ])),
                        'advantage_validation_strategy_tv': float(np.mean([
                            player['strategy_tv'] for player in validation['advantage']
                        ])),
                    },
                )
                print(
                    f'[snapshot] train={measured_training_s / 60:.1f}m '
                    f'iter={trainer.iteration} pending={evaluator.pending_count}'
                )
                while next_evaluation_s <= measured_training_s:
                    next_evaluation_s += EVALUATE_EVERY_MINUTES * 60.0

            if measured_training_s >= next_checkpoint_s:
                checkpoint_start = time.perf_counter()
                atomic_checkpoint(trainer, CHECKPOINT_PATH)
                summary = checkpoint_summary(trainer, measured_training_s)
                summary['checkpoint_s'] = time.perf_counter() - checkpoint_start
                checkpoint_records.append(summary)
                print(
                    f"[checkpoint] iter={trainer.iteration} "
                    f"size={summary['checkpoint_GiB']:.2f} GiB "
                    f"time={summary['checkpoint_s']:.1f}s"
                )
                while next_checkpoint_s <= measured_training_s:
                    next_checkpoint_s += CHECKPOINT_EVERY_MINUTES * 60.0

        # Always leave a final playable policy and exact-resume checkpoint.
        save_policy(trainer.average_policy(), str(FINAL_POLICY_DIR))
        atomic_checkpoint(trainer, CHECKPOINT_PATH)
        append_jsonl(EXACT_RESULTS_JSONL, evaluator.wait())
    finally:
        evaluator.close(wait=True)

    manifest.update({
        'status': 'complete',
        'iterations': trainer.iteration,
        'measured_training_s': measured_training_s,
        'actual_wall_s': time.perf_counter() - actual_start,
        'checkpoint': str(CHECKPOINT_PATH),
        'policy': str(FINAL_POLICY_DIR),
    })
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Exact evaluations:', len(load_jsonl(EXACT_RESULTS_JSONL)))

## Exact-resume verification

This reloads the complete trainer state without advancing it. A future session can continue by calling `run_iteration` on `resumed`.

In [ ]:
resumed = DeepCFRTrainer.load_checkpoint(CHECKPOINT_PATH, device=device)
assert resumed.iteration == trainer.iteration
assert resumed.advantage_hidden_sizes == (2048, 2048)
assert resumed.strategy_hidden_sizes == (512, 512)
assert [buffer.seen for buffer in resumed.advantage_buffers] == [
    buffer.seen for buffer in trainer.advantage_buffers
]
assert [buffer.seen for buffer in resumed.strategy_buffers] == [
    buffer.seen for buffer in trainer.strategy_buffers
]
print('Exact resume verified at iteration', resumed.iteration)
del resumed
torch.cuda.empty_cache()

In [ ]:
training_df = pd.DataFrame(load_jsonl(TRAINING_JSONL))
exact_df = pd.DataFrame(load_jsonl(EXACT_RESULTS_JSONL)).sort_values('measured_training_s')

summary = pd.DataFrame([{
    'iterations completed': int(training_df['iteration'].iloc[-1]),
    'measured training min': training_df['measured_training_s'].iloc[-1] / 60.0,
    'final exploitability': exact_df['exploitability'].iloc[-1],
    'best exploitability': exact_df['exploitability'].min(),
    'mean traversal s': np.mean([row['traversal_s'] for row in training_df['timing']]),
    'mean advantage fit s': np.mean([row['advantage_training_s'] for row in training_df['timing']]),
    'mean strategy fit s': np.mean([row['strategy_training_s'] for row in training_df['timing']]),
    'mean dense compile s': exact_df['dense_compile_s'].mean(),
    'mean exact BR s': exact_df['exact_br_s'].mean(),
    'checkpoint GiB': CHECKPOINT_PATH.stat().st_size / 2**30,
}])
display(summary.style.format(precision=6))
display(exact_df[[
    'iter', 'measured_training_s', 'exploitability',
    'advantage_validation_mse', 'advantage_validation_strategy_tv',
    'dense_compile_s', 'exact_br_s', 'evaluation_s',
]])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x = exact_df['measured_training_s'] / 60.0
axes[0].plot(x, exact_df['exploitability'], marker='o')
axes[1].plot(x, exact_df['advantage_validation_mse'], marker='o')
axes[2].plot(x, exact_df['advantage_validation_strategy_tv'], marker='o')
for ax, title, ylabel in [
    (axes[0], 'Learned-average exact exploitability', 'Exploitability'),
    (axes[1], 'Held-out advantage MSE', 'MSE'),
    (axes[2], 'Held-out induced-strategy TV', 'TV'),
]:
    ax.set(title=title, xlabel='Measured GPU-training minutes', ylabel=ylabel)
    ax.grid(True, alpha=0.3)
axes[0].set_yscale('log')
fig.tight_layout();